# House Prices: Advanced Regression Techniques
## Phase A3 Baseline Submission

**Model**: LightGBM with Optuna-tuned hyperparameters  
**Expected CV RMSLE**: 0.10353 (±0.00682)  
**Features**: 210 (with preprocessing and engineered features)

**Preprocessing**:
- Semantic imputation for missing values
- Ordinal encoding for quality features
- One-hot encoding for categorical features
- Engineered features: TotalSF, HouseAge, RemodAge, TotalBath, PorchArea

## Cell 1: Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print('Imports successful!')

## Cell 2: Helper Functions

In [ ]:
def rmsle(y_true, y_pred):
    """Calculate Root Mean Squared Logarithmic Error"""
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('Helper functions defined!')

## Cell 3: AmesPreprocessor Class (Inline)

In [ ]:
class AmesPreprocessor:
    """
    Comprehensive preprocessing for Ames Housing data
    
    Features:
    - Semantic imputation (NA meanings)
    - Grouped imputation (LotFrontage by Neighborhood)
    - Ordinal encoding for quality features
    - One-hot encoding for categorical features
    - Engineered features (TotalSF, HouseAge, etc.)
    """
    
    def __init__(self):
        self.numeric_impute_values = {}
        self.neighborhood_lot_frontage = {}
        self.feature_columns = None
        self.fitted = False
    
    def fit_transform(self, df):
        """Fit preprocessor and transform training data"""
        df = df.copy()
        df = self._create_engineered_features(df)
        df = self._handle_missing_values(df, fit=True)
        df = self._encode_quality_features(df)
        df = self._encode_categorical_features(df)
        self.feature_columns = df.columns.tolist()
        self.fitted = True
        return df
    
    def transform(self, df):
        """Transform test data using fitted parameters"""
        if not self.fitted:
            raise ValueError("Preprocessor must be fitted before transform")
        df = df.copy()
        df = self._create_engineered_features(df)
        df = self._handle_missing_values(df, fit=False)
        df = self._encode_quality_features(df)
        df = self._encode_categorical_features(df)
        
        # Align columns with training set
        for col in self.feature_columns:
            if col not in df.columns:
                df[col] = 0
        df = df[self.feature_columns]
        return df
    
    def _create_engineered_features(self, df):
        """Create domain-specific engineered features"""
        df['TotalSF'] = (
            df['TotalBsmtSF'].fillna(0) +
            df['1stFlrSF'].fillna(0) +
            df['2ndFlrSF'].fillna(0)
        )
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
        df['TotalBath'] = (
            df['FullBath'].fillna(0) +
            0.5 * df['HalfBath'].fillna(0) +
            df['BsmtFullBath'].fillna(0) +
            0.5 * df['BsmtHalfBath'].fillna(0)
        )
        df['PorchArea'] = (
            df['OpenPorchSF'].fillna(0) +
            df['EnclosedPorch'].fillna(0) +
            df['3SsnPorch'].fillna(0) +
            df['ScreenPorch'].fillna(0)
        )
        return df
    
    def _handle_missing_values(self, df, fit=False):
        """Handle missing values with semantic and grouped imputation"""
        na_means_none = [
            'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
            'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
            'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'MasVnrType'
        ]
        for col in na_means_none:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        
        na_means_zero = [
            'GarageYrBlt', 'GarageArea', 'GarageCars',
            'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
            'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
        ]
        for col in na_means_zero:
            if col in df.columns:
                df[col] = df[col].fillna(0)
        
        # LotFrontage: Grouped imputation by Neighborhood
        if 'LotFrontage' in df.columns:
            if fit:
                self.neighborhood_lot_frontage = df.groupby('Neighborhood')['LotFrontage'].median().to_dict()
            df['LotFrontage'] = df.apply(
                lambda row: self.neighborhood_lot_frontage.get(row['Neighborhood'], 0)
                if pd.isna(row['LotFrontage']) else row['LotFrontage'],
                axis=1
            )
        
        # Remaining numeric features: median imputation
        numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
        if fit:
            for col in numeric_cols:
                if df[col].isna().sum() > 0:
                    self.numeric_impute_values[col] = df[col].median()
        for col, val in self.numeric_impute_values.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        
        # Remaining categorical features: mode imputation
        categorical_cols = df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if df[col].isna().sum() > 0:
                df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'None')
        return df
    
    def _encode_quality_features(self, df):
        """Ordinal encoding for quality features"""
        qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
        qual_features = [
            'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
            'HeatingQC', 'KitchenQual', 'FireplaceQu',
            'GarageQual', 'GarageCond', 'PoolQC'
        ]
        for col in qual_features:
            if col in df.columns:
                df[col] = df[col].map(qual_map).fillna(0).astype(int)
        
        if 'BsmtExposure' in df.columns:
            bsmt_exp_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
            df['BsmtExposure'] = df['BsmtExposure'].map(bsmt_exp_map).fillna(0).astype(int)
        
        bsmt_fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
        if 'BsmtFinType1' in df.columns:
            df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_fin_map).fillna(0).astype(int)
        if 'BsmtFinType2' in df.columns:
            df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_fin_map).fillna(0).astype(int)
        
        if 'Functional' in df.columns:
            func_map = {
                'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4,
                'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0
            }
            df['Functional'] = df['Functional'].map(func_map).fillna(7).astype(int)
        
        if 'GarageFinish' in df.columns:
            garage_fin_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
            df['GarageFinish'] = df['GarageFinish'].map(garage_fin_map).fillna(0).astype(int)
        return df
    
    def _encode_categorical_features(self, df):
        """One-hot encoding for categorical features"""
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        if len(categorical_cols) > 0:
            df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
        return df

print('AmesPreprocessor class defined!')

## Cell 4: Load Data (Kaggle Paths)

In [ ]:
# Environment detection: Kaggle or local
import os

if os.path.exists('/kaggle/input'):
    # Kaggle environment
    DATA_PATH = '/kaggle/input/house-prices-advanced-regression-techniques'
    OUTPUT_PATH = '/kaggle/working'
    print('Running in Kaggle environment')
else:
    # Local environment - handle different working directories
    if os.path.exists('data/train.csv'):
        DATA_PATH = 'data'
        OUTPUT_PATH = 'outputs/predictions'
    elif os.path.exists('../data/train.csv'):
        DATA_PATH = '../data'
        OUTPUT_PATH = '../outputs/predictions'
    else:
        # Assume we're in the project root
        project_root = os.path.abspath(os.path.join(os.path.dirname(__file__ if '__file__' in dir() else '.'), '..'))
        DATA_PATH = os.path.join(project_root, 'data')
        OUTPUT_PATH = os.path.join(project_root, 'outputs', 'predictions')
    
    print(f'Running in local environment')
    print(f'DATA_PATH: {DATA_PATH}')

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load data
train = pd.read_csv(f'{DATA_PATH}/train.csv')
test = pd.read_csv(f'{DATA_PATH}/test.csv')

print(f'\nTrain shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'\nTarget (SalePrice) stats:')
print(f'  Mean: ${train["SalePrice"].mean():,.2f}')
print(f'  Median: ${train["SalePrice"].median():,.2f}')
print(f'  Std: ${train["SalePrice"].std():,.2f}')

## Cell 5: Preprocessing

In [ ]:
# Extract target and log-transform
target = train['SalePrice']
target_log = np.log1p(target)
train_ids = train['Id']
test_ids = test['Id']

print(f'Target (log-transformed) stats:')
print(f'  Mean: {target_log.mean():.4f}')
print(f'  Std: {target_log.std():.4f}')

# Preprocessing
preprocessor = AmesPreprocessor()
X_train_raw = train.drop(['Id', 'SalePrice'], axis=1)
X_test_raw = test.drop(['Id'], axis=1)

print('\nFitting preprocessor on training data...')
X_train = preprocessor.fit_transform(X_train_raw)
print(f'Training data shape after preprocessing: {X_train.shape}')

print('Transforming test data...')
X_test = preprocessor.transform(X_test_raw)
print(f'Test data shape after preprocessing: {X_test.shape}')

print(f'\nTotal features: {X_train.shape[1]}')
print('Features include:')
print('  - Engineered: TotalSF, HouseAge, RemodAge, TotalBath, PorchArea')
print('  - Ordinal encoded quality features')
print('  - One-hot encoded categorical features')

## Cell 6: Model Parameters (Optuna-tuned)

In [ ]:
# Best hyperparameters from Optuna tuning (30 trials, 5-fold CV)
# Local CV RMSLE: 0.10353 (±0.00682)
model_params = {
    'n_estimators': 800,
    'learning_rate': 0.04435737213089749,
    'max_depth': 13,
    'num_leaves': 69,
    'min_child_samples': 35,
    'subsample': 0.6173983977980623,
    'colsample_bytree': 0.6241701198264097,
    'reg_alpha': 0.015936847132187043,
    'reg_lambda': 6.573787083889792e-08,
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'random_state': 42
}

print('Model parameters loaded!')
print(f'  n_estimators: {model_params["n_estimators"]}')
print(f'  learning_rate: {model_params["learning_rate"]:.4f}')
print(f'  max_depth: {model_params["max_depth"]}')
print(f'  num_leaves: {model_params["num_leaves"]}')

## Cell 7: Cross-Validation

In [ ]:
print('Running 5-Fold Cross-Validation...\n')

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
cv_scores_rmsle = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f'Fold {fold}/{n_folds}:', end=' ')
    
    # Split data
    X_train_fold = X_train.iloc[train_idx]
    X_val_fold = X_train.iloc[val_idx]
    y_train_fold = target_log.iloc[train_idx]
    y_val_fold = target_log.iloc[val_idx]
    y_val_original = target.iloc[val_idx]
    
    # Train model
    model = lgb.LGBMRegressor(**model_params)
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Predict and calculate RMSLE
    val_pred_log = model.predict(X_val_fold)
    val_pred = np.expm1(val_pred_log)
    fold_rmsle = rmsle(y_val_original, val_pred)
    cv_scores_rmsle.append(fold_rmsle)
    
    print(f'RMSLE = {fold_rmsle:.5f}')

# Summary
mean_rmsle = np.mean(cv_scores_rmsle)
std_rmsle = np.std(cv_scores_rmsle)

print('\n' + '='*50)
print(f'Cross-Validation Results ({n_folds}-Fold):')
print('='*50)
print(f'Mean RMSLE: {mean_rmsle:.5f} (± {std_rmsle:.5f})')
print(f'Individual Folds: {[f"{s:.5f}" for s in cv_scores_rmsle]}')
print('='*50)

## Cell 8: Train Final Model on Full Dataset

In [ ]:
print('Training final model on full training data...')

final_model = lgb.LGBMRegressor(**model_params)
final_model.fit(X_train, target_log)

print('Final model trained successfully!')

## Cell 9: Generate Predictions

In [ ]:
print('Making predictions on test set...')

test_pred_log = final_model.predict(X_test)
test_pred = np.expm1(test_pred_log)  # Transform back from log space

print(f'Predictions stats:')
print(f'  Mean: ${test_pred.mean():,.2f}')
print(f'  Median: ${np.median(test_pred):,.2f}')
print(f'  Min: ${test_pred.min():,.2f}')
print(f'  Max: ${test_pred.max():,.2f}')

## Cell 10: Create and Save Submission

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

# Save to appropriate directory
submission.to_csv(f'{OUTPUT_PATH}/submission.csv', index=False)

print(f'Submission file saved to {OUTPUT_PATH}/submission.csv')
print(f'\nSubmission shape: {submission.shape}')
print('\nFirst 10 rows:')
print(submission.head(10))

## Cell 11: Summary

In [ ]:
print('='*60)
print('BASELINE SUBMISSION COMPLETE!')
print('='*60)
print(f'Model: LightGBM (Optuna-tuned)')
print(f'Metric: RMSLE (Root Mean Squared Logarithmic Error)')
print(f'Cross-Validation: {n_folds}-Fold')
print(f'Mean CV RMSLE: {mean_rmsle:.5f} (± {std_rmsle:.5f})')
print(f'Features used: {X_train.shape[1]}')
print(f'Submission file: /kaggle/working/submission.csv')
print('='*60)
print('\nNext steps:')
print('1. Click "Save Version" → "Save & Run All"')
print('2. Wait for execution to complete')
print('3. Click "Submit to Competition"')
print('4. Check leaderboard for your score!')
print('='*60)